# Limpieza y Transformación (ETL) del año 2020
Usando el dataframe ya en limpio del año 2019 decidi contraponer el año 2020 y 2021 para luego al final, cuando cree visualizaciones, poder tener un contexto de pre pandemia, pandemia en si y post pandemia para medir.

In [11]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) # muestra todas las columnas
pd.set_option("display.width", 120) # ajusto el ancho


path_clean = "../data/clean/"
df_2019_clean = pd.read_csv(path_clean + "historico_2019_clean.csv")
schema_2019 = df_2019_clean.columns.tolist()
schema_2019

['periodo',
 'fecha',
 'desde',
 'hasta',
 'linea',
 'molinete',
 'estacion',
 'pax_pagos',
 'pax_pases_pagos',
 'pax_franq',
 'total',
 'hora_desde',
 'hora_hasta',
 'dia_semana',
 'mes',
 'dia_mes',
 'es_fin_semana']

# IMPORTANTE: DATA SCHEMA
Opte por usar el df limpio del 2019 como esquema inicial para respetar la estructura de datos presentada en aquel archivo. Tanto en este file como en el próximo a crear , año 2021, tendran que respetar las columnas de datos propuestas por el 2019 y en caso de tener inconsistencias, completar como lo amerite.


In [12]:
#Carga datos crudos

path_raw = "../data/raw/"
df_2020 = pd.read_csv(path_raw + "historico_2020.csv")

df_2020.shape, df_2020.columns.tolist()


((5781006, 10),
 ['FECHA',
  'DESDE',
  'HASTA',
  'LINEA',
  'MOLINETE',
  'ESTACION',
  'pax_pagos',
  'pax_pases_pagos',
  'pax_franq',
  'pax_TOTAL'])

In [13]:
cols_2020 = set(df_2020.columns)
cols_2019 = set(schema_2019)

faltan_en_2020 = sorted(list(cols_2019 - cols_2020))
sobran_en_2020 = sorted(list(cols_2020 - cols_2019))

faltan_en_2020, sobran_en_2020


(['desde',
  'dia_mes',
  'dia_semana',
  'es_fin_semana',
  'estacion',
  'fecha',
  'hasta',
  'hora_desde',
  'hora_hasta',
  'linea',
  'mes',
  'molinete',
  'periodo',
  'total'],
 ['DESDE', 'ESTACION', 'FECHA', 'HASTA', 'LINEA', 'MOLINETE', 'pax_TOTAL'])

### Diferencias estructurales entre los datasets 2019 y 2020

Durante la comparación del esquema del dataset correspondiente al año 2020 con
el esquema de referencia definido a partir de 2019, noté diferencias
estructurales relevantes que justifican la implementación de un proceso ETL
específico para este período.

En particular, el dataset 2020 presenta:
- Nombres de columnas en mayúsculas (`FECHA`, `LINEA`, `ESTACION`, `MOLINETE`),
  a diferencia del formato estandarizado en 2019.
- Un cambio en la nomenclatura de la métrica agregada de pasajeros, donde
  `total` en 2019 aparece como `pax_TOTAL` en 2020.
- Ausencia de columnas derivadas presentes en el dataset limpio de 2019
  (por ejemplo: `periodo`, `hora_desde`, `hora_hasta`, `dia_semana`, `mes`,
  `dia_mes`, `es_fin_semana`), las cuales no forman parte del dataset crudo
  y deben ser generadas durante la etapa de transformación.

Estas diferencias no implican pérdida de información, sino variaciones en el
formato y estructura del dataset, probablemente asociadas a cambios en los
procesos de generación o publicación de los datos durante el período pandémico.

Por este motivo, adopto un enfoque de limpieza y transformación específico
para el año 2020, utilizando el esquema del dataset 2019 como referencia
estructural.
Este enfoque me permite alinear los datos a un formato común,
garantizando consistencia y comparabilidad temporal sin forzar información
inexistente.


In [14]:
rename_map = {
    "FECHA": "fecha",
    "DESDE": "desde",
    "HASTA": "hasta",
    "LINEA": "linea",
    "MOLINETE": "molinete",
    "ESTACION": "estacion",
    "pax_TOTAL": "total",
}

df_2020 = df_2020.rename(columns=rename_map)
df_2020.columns.tolist()


['fecha',
 'desde',
 'hasta',
 'linea',
 'molinete',
 'estacion',
 'pax_pagos',
 'pax_pases_pagos',
 'pax_franq',
 'total']

In [15]:
cols_2020 = set(df_2020.columns)
cols_2019 = set(schema_2019)

faltan_en_2020 = sorted(list(cols_2019 - cols_2020))
sobran_en_2020 = sorted(list(cols_2020 - cols_2019))

faltan_en_2020, sobran_en_2020


(['dia_mes',
  'dia_semana',
  'es_fin_semana',
  'hora_desde',
  'hora_hasta',
  'mes',
  'periodo'],
 [])

### Alineación del esquema 2020 con el dataset de referencia 2019

Luego de estandarizar los nombres de columnas del dataset 2020, hice una
comparación contra el esquema de referencia definido a partir del dataset limpio
de 2019.

El análisis muestra que:
- No existen columnas adicionales en el dataset 2020 que no estén presentes en
  el esquema de referencia.
- Las únicas columnas ausentes corresponden a variables derivadas
  (`periodo`, `mes`, `dia_mes`, `dia_semana`, `es_fin_semana`,
  `hora_desde`, `hora_hasta`), las cuales no forman parte del dataset crudo y
  deben ser generadas durante la etapa de transformación.

Este resultado confirma la compatibilidad estructural entre los datasets y
valida el uso de un proceso ETL específico para el año 2020, orientado a la
creación de variables derivadas y a la alineación final del esquema, garantizando
consistencia y comparabilidad temporal con el período pre-pandemia.


In [16]:
df_2020["fecha"] = pd.to_datetime(df_2020["fecha"], errors="coerce")

df_2020[["desde","hasta"]].head(5)

,desde,hasta
0,08:00:00,08:15:00
1,08:00:00,08:15:00
2,08:00:00,08:15:00
3,08:00:00,08:15:00
4,08:00:00,08:15:00


In [19]:
df_2020["linea"] = (
    df_2020["linea"].astype("string").str.strip().str.upper().astype("category")
)

df_2020["estacion"] = (
    df_2020["estacion"].astype("string").str.strip().str.title().astype("category")
)

df_2020["molinete"] = (
    df_2020["molinete"].astype("string").str.strip().str.upper().astype("category")
)
